## Question 1

Run it to test that it's working locally:

```bash
    docker run -it --rm -p 9696:9696 zoomcamp-model:3.13.10-hw10
```

And in another terminal, execute q6_test.py file:

```python
    python q6_test.py
```

You should see this:

```python
    {'conversion_probability': value, 'conversion': False}
```

Here value is the probability of getting a subscription. You need to choose the right one.

In [ ]:
{
    'conversion_probability': 0.49999999999842815, 
    'conversion': False
}

Before you continue setup kubectl and kind

## Question 2. 

Version of kind is: 0.30.0

## Create a Kind Cluster

Let's create a local Kubernetes cluster: 

```bash
    sudo kind create cluster --name hw-10
```
I named mine hw-10

Verify cluster is running

```bash
    sudo kubectl cluster-info

    sudo kubectl get nodes
```

The status of the node has to be READY

## Question 3

What's the smallest deployable computing unit that we can create 
and manage in Kubernetes (kind in our case)?

Is Pod

Initialize an uv environment and add dependencies

```bash
    uv add fastapi uvicorn onnxruntime keras-image-helper numpy
```

## Question 4

Now let's test if everything works. Use kubectl to get the list of running services.

```bash
    sudo kubectl get services
```

What's the Type of the service that is already running there?

The type is ClusterIP

## Question 5

To be able to use the docker image we previously created (zoomcamp-model:3.13.10-hw10), we need to register it with kind.

What's the command we need to run for that?

The command is:

```bash
    sudo kind load docker-image docker_image:tag --name cluster_name
```

(docker-image, docker_image:tag, cluster_name values have to be replaced to match existing ones)

## Question 6

Now let's create a deployment config (e.g. deployment.yaml):

Replace Image, Memory, CPU, Port values with the correct values.

What is the value for Port?

Te value I used is the same as in Dockerfile

```yaml
    EXPORT <PORT>
```

In lecture it was 8080, for downloaded model in Q1 it was 9696 so it may be different depending on which port you used in your Dockerfile. I set it to 4444 to test locally.

## Question 7

Let's create a service for this deployment (service.yaml)

What do we need to write instead of app: ???

You have to write the name of the app, as reference I used containers->name which comes before containers->image in deployment.yaml file

```yaml
    app: app_name
```

Replace app_name with the name you defined for your service

## Testing the service

We can test our service locally by forwarding the port 9696 on our computer to the port 80 on the service:

```bash
    sudo kubectl port-forward service/app_name 9696:80
```

Here app_name is the name of your service

Mine uses port 4444 as i cannot use port 80 or 8080 as they are used by other processes

Run q6_test.py (from the homework 5) once again to verify that everything is working. You should get the same result as in Question 1.

## Autoscaling

Use the following command to create the HPA:

```bash
    sudo kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3
```

Ypu may notice: Flag --cpu-percent has been deprecated, Use --cpu with percentage or resource quantity format (e.g., '70%' for utilization or '500m' for milliCPU).

```bash
    sudo kubectl autoscale deployment subscription --name subscription-hpa --cpu=20% --min=1 --max=3
```

You can check the current status of the new HPA by running: 

```bash
    sudo kubectl get hpa
```

Note: In case the HPA instance doesn't run properly, try to install the latest Metrics Server release from the components.yaml manifest:

```bash
    sudo kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
```

In case TARGETS column shows:

```bash
    cpu: <unknown>/20%
```

Remember that in lectures you have to set up the metrics server

Then use it without TLS as it is not setup out of the box to work without it

In my case I created an hpa.yaml file using the previous HPA command as reference

Once the metrics server has been set up apply the hpa.yaml file configuration:

```bash
    sudo kubectl apply -f path/to/hpa.yaml
```

Use:

```bash
    sudo kubectl get hpa
```

and check if the TARGETS column is now showing correct numerical value:

```bash
    cpu: 0%/20%
```

Or similar instead of unknown

If it is not working delete all resourcess related to your kubernetes cluster, then delete the cluster and go step by step again from cluster creation and so on, this worked for me to spot what was causing the get hpa TARGETS column showing unknown

## Increase the load

Let's see how the autoscaler reacts to increasing the load. To do this, we can slightly modify the existing q6_test.py script by putting the operator that sends the request to the subscription service into a loop.

~~~python
while True:
    sleep(0.1)
    response = requests.post(url, json=client).json()
    print(response)
~~~

Now you can run this script.

## Question 8 (optional)

Run

```bash
    sudo kubectl get hpa subscription-hpa --watch
```

command to monitor how the autoscaler performs. Within a minute or so, you should see the higher CPU load; and then - more replicas. What was the maximum amount of the replicas during this test?

Note: It may take a few minutes to stabilize the number of replicas. Since the amount of load is not controlled in any way it may happen that the final number of replicas will differ from initial.

The maximum amount of replicas behaved this way:

first test: sleep 0.1, Targets 0%/20%, replicas 1

first test: sleep 0.1, Targets 3%/20%, replicas 1

first test: sleep 0.1, Targets 6%/20%, replicas 1

first test: sleep 0.1, Targets 6%/20%, replicas 1

first test: sleep 0.1, Targets 6%/20%, replicas 1

----------------------------------------------------
second test: sleep 0.01, Targets 18%/20%, replicas 1

second test: sleep 0.01, Targets 42%/20%, replicas 1

second test: sleep 0.01, Targets 42%/20%, replicas 3

second test: sleep 0.01, Targets 14%/20%, replicas 3

second test: sleep 0.01, Targets 14%/20%, replicas 3

So not changing the sleep amount (0.1) replicas stayed at 1

Changing the sleep amount (0.01) replicas reached 3 and stayed at 3